# 07 · Validación multisemilla RT-DETR Estados Unidos

Detector activo: **RT-DETRv2-S** con **OpenVINO Runtime nativo en Python**, checkpoint `merve/license-plates-rtdetrv2` con licencia Apache-2.0.

In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "optuna>=4,<5" "fast-plate-ocr==1.1.0" "roboflow==1.7.1" "openvino>=2025.4,<2027"

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_ROOT=Path("/content/SIRAP_USA_ECUADOR")
if not REPO_ROOT.exists(): subprocess.run(["git","clone","--depth","1","https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git",str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT); sys.path.insert(0,str(REPO_ROOT))
COUNTRY="usa"
MODEL_ID="merve/license-plates-rtdetrv2"
print("Perfil:",COUNTRY,"| RT-DETRv2-S:",MODEL_ID)


## Reproducibilidad entre semillas

Cada semilla guarda su checkpoint y resultado de validación usando la receta USA: hasta **50 épocas**, paciencia **8**, batch 8 y learning rate 5e-5. La corrida de tres semillas es costosa y queda desactivada de forma predeterminada. No uses el test independiente para elegir hiperparámetros.

In [ ]:
from pathlib import Path
import pandas as pd
from src.train import train_detector
from src.evaluate import evaluate_detector
DATA_DIR=None; SEEDS=[0,1,2]; RUN=False
if RUN:
    if not DATA_DIR: raise ValueError("Define DATA_DIR")
    rows=[]
    for seed in SEEDS:
        out=Path(f"/content/rtdetr_usa/seed_{seed}/checkpoint")
        train_detector(DATA_DIR,"usa",MODEL_ID,out,out.parent/"trainer",epochs=50,early_stopping_patience=8,batch_size=8,
                       learning_rate=5e-5,seed=seed,force=True)
        metrics=evaluate_detector(DATA_DIR,"usa","val",out,MODEL_ID,threshold=0.001,
            result_path=f"/content/rtdetr_usa/seed_{seed}_val.json")
        rows.append({"seed":seed,"map50_95":metrics["map50_95"],"map50":metrics["map50"],
                     "precision":metrics["precision"],"recall":metrics["recall"]})
    result=pd.DataFrame(rows)
    result.to_csv("/content/validacion_multisemilla_rtdetr_usa.csv",index=False)
    display(result)
else: print("Corrida desactivada; define dataset y presupuesto antes de ejecutarla.")
